## Dataset Creation

First, I will set up the group number and random seeds for reproducibility, then calculate the number of medicines and their hidden success probabilities.

In [15]:
import random
import numpy as np
import pandas as pd

# Group number (replace with your actual group number)
G = 10 # Example group number for demonstration

# Set random seeds for reproducibility
random.seed(G)
np.random.seed(G)

print(f"Group Number (G): {G}")

Group Number (G): 10


### 1.1 Number of Medicines

Calculate the number of available treatment choices (K).

In [16]:
num_medicines = (G % 3) + 5
print(f"Number of medicines (K): {num_medicines}")

Number of medicines (K): 6


### 1.2 Hidden Success Probability

Calculate the hidden probability of successful recovery for each medicine.

In [17]:
success_probabilities = [0.4 + ((G + i) % 6) * 0.07 for i in range(num_medicines)]

for i, prob in enumerate(success_probabilities):
    print(f"Medicine {i}: Success Probability = {prob:.2f}")

Medicine 0: Success Probability = 0.68
Medicine 1: Success Probability = 0.75
Medicine 2: Success Probability = 0.40
Medicine 3: Success Probability = 0.47
Medicine 4: Success Probability = 0.54
Medicine 5: Success Probability = 0.61


### 1.3 Patient Severity and Utility Logic & 1.4 Dataset Schema

Generate exactly 1000 patient records, including their `patient_id` and `severity_score`.
The `assigned_medicine`, `clinical_outcome`, and `utility_score` columns will be dynamically populated during algorithm execution.

In [18]:
num_patients = 1000
patient_data = []

for patient_id in range(num_patients):
    severity_score = (patient_id % 5) + 1
    patient_data.append({
        'patient_id': patient_id,
        'severity_score': severity_score
    })

# Create the initial DataFrame
patient_df = pd.DataFrame(patient_data)

print(f"Generated {len(patient_df)} patient records.")
display(patient_df.head())

Generated 1000 patient records.


,patient_id,severity_score
0,0,1
1,1,2
2,2,3
3,3,4
4,4,5


## **Task 1: Dataset Design**

### 1. Generate your synthetic environment.

The synthetic environment, including patient records and medicine parameters, has been generated in the preceding cells. This involved setting the group number, random seeds, calculating the number of medicines (K), their hidden success probabilities, and creating 1000 patient records with severity scores.

### 2. Display: group number G, total medicines K, hidden success probabilities of all medicines

In [19]:
print(f"Group Number (G): {G}")
print(f"Total Medicines (K): {num_medicines}")
print("Hidden Success Probabilities of all Medicines:")
for i, prob in enumerate(success_probabilities):
    print(f"  Medicine {i}: {prob:.2f}")

Group Number (G): 10
Total Medicines (K): 6
Hidden Success Probabilities of all Medicines:
  Medicine 0: 0.68
  Medicine 1: 0.75
  Medicine 2: 0.40
  Medicine 3: 0.47
  Medicine 4: 0.54
  Medicine 5: 0.61


### 3. Print first 10 dataset rows.

In [20]:
print("First 10 rows of the generated patient dataset:")
display(patient_df.head(10))

First 10 rows of the generated patient dataset:


,patient_id,severity_score
0,0,1
1,1,2
2,2,3
3,3,4
4,4,5
5,5,1
6,6,2
7,7,3
8,8,4
9,9,5


### Helper Function to Simulate Patient Treatment

Before implementing the MAB algorithms, I'll create a helper function `simulate_treatment` that takes a medicine index and a patient's severity score, then calculates the `clinical_outcome` and `utility_score` based on the defined rules.

In [21]:
def simulate_treatment(medicine_idx, severity_score, success_probabilities):
    """
    Simulates the clinical outcome and utility score for a given patient and medicine.

    Args:
        medicine_idx (int): The index of the chosen medicine.
        severity_score (int): The severity score of the patient (1-5).
        success_probabilities (list): List of hidden success probabilities for each medicine.

    Returns:
        tuple: A tuple containing (clinical_outcome, utility_score).
    """
    # Get the success probability for the chosen medicine
    p_success = success_probabilities[medicine_idx]

    # Determine clinical outcome (1 for recovered, 0 for not recovered)
    clinical_outcome = 1 if np.random.rand() < p_success else 0

    # Calculate utility score
    # Interpretation:
    # If patient recovers and severity = 1 -> reward = 0.9
    # If patient recovers and severity = 5 -> reward = 0.5
    # If patient does not recover -> reward = 0
    if clinical_outcome == 1:
        utility_score = 1 - (severity_score / 10)
    else:
        utility_score = 0

    return clinical_outcome, utility_score

print("Helper function 'simulate_treatment' defined.")

Helper function 'simulate_treatment' defined.


## **Task 2: Immediate Exploitation Strategy**

This strategy suggests that "Once a treatment appears best, continue prescribing only that treatment for all future patients." We will implement this by initially testing each medicine a fixed number of times (10 in this case), identifying the one with the highest observed success rate, and then exclusively using that medicine for the rest of the patients.

In [22]:
def run_immediate_exploitation(num_medicines, success_probabilities, patient_df, initial_test_rounds=10):
    # Initialize counts and estimated values for each medicine
    counts = np.zeros(num_medicines)  # Number of times each medicine has been pulled
    rewards_sum = np.zeros(num_medicines) # Sum of clinical outcomes (successes) for each medicine
    estimated_success_rates = np.zeros(num_medicines) # Average clinical outcome (success rate)

    # DataFrame to store results for this run
    results_df = patient_df.copy()
    results_df['assigned_medicine'] = -1
    results_df['clinical_outcome'] = -1
    results_df['utility_score'] = 0.0

    cumulative_reward = 0
    best_medicine_to_exploit = -1 # Will be determined after initial test rounds

    print(f"\n--- Running Immediate Exploitation Strategy (Initial test rounds: {initial_test_rounds}) ---")

    for i, row in patient_df.iterrows():
        patient_id = row['patient_id']
        severity_score = row['severity_score']

        if i < num_medicines * initial_test_rounds: # Initial testing phase
            # Cycle through medicines for initial testing
            chosen_medicine = i % num_medicines

            clinical_outcome, utility_score = simulate_treatment(
                chosen_medicine, severity_score, success_probabilities
            )

            # Update bandit statistics based on clinical outcome (for determining best arm)
            counts[chosen_medicine] += 1
            rewards_sum[chosen_medicine] += clinical_outcome
            # Avoid division by zero if counts[chosen_medicine] is 0
            estimated_success_rates[chosen_medicine] = rewards_sum[chosen_medicine] / counts[chosen_medicine] if counts[chosen_medicine] > 0 else 0

        else: # Exploitation phase
            # If it's the first patient in the exploitation phase, determine the best medicine
            if best_medicine_to_exploit == -1: # Only once after initial testing
                best_medicine_to_exploit = np.argmax(estimated_success_rates)
                print(f"Initial testing complete. Best medicine identified: {best_medicine_to_exploit} (Estimated Success Rate: {estimated_success_rates[best_medicine_to_exploit]:.2f})")

            chosen_medicine = best_medicine_to_exploit # Always exploit the best one

            clinical_outcome, utility_score = simulate_treatment(
                chosen_medicine, severity_score, success_probabilities
            )
            # No further updates to counts/rewards_sum as we are only exploiting

        # Record results
        results_df.loc[i, 'assigned_medicine'] = chosen_medicine
        results_df.loc[i, 'clinical_outcome'] = clinical_outcome
        results_df.loc[i, 'utility_score'] = utility_score

        cumulative_reward += utility_score

    print("Immediate Exploitation simulation complete.")
    print(f"Total Cumulative Utility Score: {cumulative_reward:.2f}")
    return results_df, cumulative_reward

# Run the simulation for Task 2
ie_results_df, ie_cumulative_reward = run_immediate_exploitation(num_medicines, success_probabilities, patient_df, initial_test_rounds=10)
display(ie_results_df.head(10))

print(f"\nCumulative Reward for Immediate Exploitation: {ie_cumulative_reward:.2f}")


--- Running Immediate Exploitation Strategy (Initial test rounds: 10) ---
Initial testing complete. Best medicine identified: 0 (Estimated Success Rate: 0.70)
Immediate Exploitation simulation complete.
Total Cumulative Utility Score: 469.00


,patient_id,severity_score,assigned_medicine,clinical_outcome,utility_score
0,0,1,0,0,0.0
1,1,2,1,1,0.8
2,2,3,2,0,0.0
3,3,4,3,0,0.0
4,4,5,4,1,0.5
5,5,1,5,1,0.9
6,6,2,0,1,0.8
7,7,3,1,0,0.0
8,8,4,2,1,0.6
9,9,5,3,1,0.5



Cumulative Reward for Immediate Exploitation: 469.00


## Multi-Armed Bandit (MAB) Algorithms Implementation (Function Definitions)

This section defines the MAB algorithms that will be used in subsequent tasks. The functions are defined here but not executed until explicitly called within the tasks.

### 1. Epsilon-Greedy Algorithm

Epsilon-Greedy is a simple yet effective MAB algorithm that chooses a random arm with a small probability (epsilon) to explore, and otherwise exploits the arm with the highest estimated reward.

In [23]:
def run_epsilon_greedy(epsilon, num_medicines, success_probabilities, patient_df):
    # Initialize counts and estimated values for each medicine
    counts = np.zeros(num_medicines)
    values = np.zeros(num_medicines)

    # DataFrame to store results for this run
    results_df = patient_df.copy()
    results_df['assigned_medicine'] = -1
    results_df['clinical_outcome'] = -1
    results_df['utility_score'] = 0.0

    cumulative_reward = 0

    print(f"\n--- Running Epsilon-Greedy (epsilon={epsilon}) ---")

    for i, row in patient_df.iterrows():
        patient_id = row['patient_id']
        severity_score = row['severity_score']

        # Choose an arm (medicine)
        if np.random.rand() < epsilon:
            chosen_medicine = random.randrange(num_medicines) # Explore
        else:
            chosen_medicine = np.argmax(values) # Exploit

        # Simulate treatment and get outcome/utility
        clinical_outcome, utility_score = simulate_treatment(
            chosen_medicine, severity_score, success_probabilities
        )

        # Update counts and values for the chosen medicine
        counts[chosen_medicine] += 1
        n = counts[chosen_medicine]
        current_value = values[chosen_medicine]
        new_value = ((n - 1) / n) * current_value + (1 / n) * clinical_outcome # Update based on clinical outcome for bandit stats
        values[chosen_medicine] = new_value

        # Record results
        results_df.loc[i, 'assigned_medicine'] = chosen_medicine
        results_df.loc[i, 'clinical_outcome'] = clinical_outcome
        results_df.loc[i, 'utility_score'] = utility_score

        cumulative_reward += utility_score

    print("Epsilon-Greedy simulation complete.")
    print(f"Total Cumulative Utility Score: {cumulative_reward:.2f}")
    return results_df, cumulative_reward

### 2. UCB1 (Upper Confidence Bound 1) Algorithm

UCB1 is an algorithm that tries to minimize regret by selecting arms based on an upper confidence bound of their value. It favors arms that have been played less or have high estimated rewards, ensuring exploration of less certain options.

In [24]:
def run_ucb1(num_medicines, success_probabilities, patient_df):
    # Initialize counts and estimated values for each medicine
    counts = np.zeros(num_medicines)  # N_i(t) - number of times arm i has been played
    values = np.zeros(num_medicines)  # Q_i(t) - estimated value of arm i (average reward)
    total_plays = 0  # t - total number of plays

    # DataFrame to store results for this run
    results_df = patient_df.copy()
    results_df['assigned_medicine'] = -1
    results_df['clinical_outcome'] = -1
    results_df['utility_score'] = 0.0

    cumulative_reward = 0

    print(f"\n--- Running UCB1 ---")

    for i, row in patient_df.iterrows():
        patient_id = row['patient_id']
        severity_score = row['severity_score']
        total_plays += 1

        # UCB1 requires each arm to be played at least once initially
        if total_plays <= num_medicines:
            chosen_medicine = total_plays - 1 # Play each medicine once in the beginning
        else:
            ucb_values = np.zeros(num_medicines)
            for medicine_idx in range(num_medicines):
                if counts[medicine_idx] == 0:
                    ucb_values[medicine_idx] = float('inf') # Ensure unplayed arms are chosen first
                else:
                    # UCB1 formula: Q_i(t) + sqrt(2 * log(t) / N_i(t))
                    exploration_term = np.sqrt(2 * np.log(total_plays) / counts[medicine_idx])
                    ucb_values[medicine_idx] = values[medicine_idx] + exploration_term
            chosen_medicine = np.argmax(ucb_values)

        # Simulate treatment and get outcome/utility
        clinical_outcome, utility_score = simulate_treatment(
            chosen_medicine, severity_score, success_probabilities
        )

        # Update counts and values for the chosen medicine
        counts[chosen_medicine] += 1
        n = counts[chosen_medicine]
        current_value = values[chosen_medicine]
        new_value = ((n - 1) / n) * current_value + (1 / n) * clinical_outcome # Update based on clinical outcome
        values[chosen_medicine] = new_value

        # Record results
        results_df.loc[i, 'assigned_medicine'] = chosen_medicine
        results_df.loc[i, 'clinical_outcome'] = clinical_outcome
        results_df.loc[i, 'utility_score'] = utility_score

        cumulative_reward += utility_score

    print("UCB1 simulation complete.")
    print(f"Total Cumulative Utility Score: {cumulative_reward:.2f}")
    return results_df, cumulative_reward

### 3. Thompson Sampling Algorithm

Thompson Sampling is a probabilistic algorithm that samples from the posterior distribution of each arm's success probability and chooses the arm with the highest sampled value. This inherently balances exploration and exploitation based on the uncertainty of each arm's reward distribution.

In [25]:
def run_thompson_sampling(num_medicines, success_probabilities, patient_df):
    # Initialize parameters for Beta distribution (conjugate prior for Bernoulli likelihood)
    # Each arm's success probability is modeled as a Beta distribution (Beta(alpha, beta))
    # alpha (successes) and beta (failures)
    alphas = np.ones(num_medicines)  # Start with Beta(1,1) - uniform prior
    betas = np.ones(num_medicines)

    # DataFrame to store results for this run
    results_df = patient_df.copy()
    results_df['assigned_medicine'] = -1
    results_df['clinical_outcome'] = -1
    results_df['utility_score'] = 0.0

    cumulative_reward = 0

    print(f"\n--- Running Thompson Sampling ---")

    for i, row in patient_df.iterrows():
        patient_id = row['patient_id']
        severity_score = row['severity_score']

        # Sample from the Beta distribution for each medicine
        sampled_probabilities = [np.random.beta(alphas[j], betas[j]) for j in range(num_medicines)]

        # Choose the medicine with the highest sampled probability
        chosen_medicine = np.argmax(sampled_probabilities)

        # Simulate treatment and get outcome/utility
        clinical_outcome, utility_score = simulate_treatment(
            chosen_medicine, severity_score, success_probabilities
        )

        # Update Beta distribution parameters based on clinical outcome
        if clinical_outcome == 1:
            alphas[chosen_medicine] += 1
        else:
            betas[chosen_medicine] += 1

        # Record results
        results_df.loc[i, 'assigned_medicine'] = chosen_medicine
        results_df.loc[i, 'clinical_outcome'] = clinical_outcome
        results_df.loc[i, 'utility_score'] = utility_score

        cumulative_reward += utility_score

    print("Thompson Sampling simulation complete.")
    print(f"Total Cumulative Utility Score: {cumulative_reward:.2f}")
    return results_df, cumulative_reward

## **Task 3: Controlled Clinical Trial Strategy**

Doctors suggest: "Most patients should receive the current best treatment, but occasionally another treatment should be tested to discover hidden opportunities." This is essentially an Epsilon-Greedy approach, which we've already implemented. We will re-use our `run_epsilon_greedy` function but evaluate it with different exploration rates (epsilon values) as requested.

### 3.1 Implement with 10% intentional exploration (epsilon = 0.1)

In [26]:
print("\n--- Task 3.1: Running Epsilon-Greedy with 10% exploration (epsilon=0.1) ---")
# Implement this strategy with 10% intentional exploration
epsilon_greedy_results_df_0_1, eg_cumulative_reward_0_1 = run_epsilon_greedy(0.1, num_medicines, success_probabilities, patient_df)
print(f"Cumulative Reward (epsilon=0.1): {eg_cumulative_reward_0_1:.2f}")


--- Task 3.1: Running Epsilon-Greedy with 10% exploration (epsilon=0.1) ---

--- Running Epsilon-Greedy (epsilon=0.1) ---
Epsilon-Greedy simulation complete.
Total Cumulative Utility Score: 493.80
Cumulative Reward (epsilon=0.1): 493.80


### 3.2 Analyze what happens if exploration changes to: 1% and 50%

#### Exploration with 1% (epsilon = 0.01)

In [27]:
print("\n--- Task 3.2: Running Epsilon-Greedy with 1% exploration (epsilon=0.01) ---")
epsilon_greedy_results_df_0_01, eg_cumulative_reward_0_01 = run_epsilon_greedy(0.01, num_medicines, success_probabilities, patient_df)
print(f"Cumulative Reward (epsilon=0.01): {eg_cumulative_reward_0_01:.2f}")


--- Task 3.2: Running Epsilon-Greedy with 1% exploration (epsilon=0.01) ---

--- Running Epsilon-Greedy (epsilon=0.01) ---
Epsilon-Greedy simulation complete.
Total Cumulative Utility Score: 510.80
Cumulative Reward (epsilon=0.01): 510.80


#### Exploration with 50% (epsilon = 0.5)

In [28]:
print("\n--- Task 3.2: Running Epsilon-Greedy with 50% exploration (epsilon=0.5) ---")
epsilon_greedy_results_df_0_5, eg_cumulative_reward_0_5 = run_epsilon_greedy(0.5, num_medicines, success_probabilities, patient_df)
print(f"Cumulative Reward (epsilon=0.5): {eg_cumulative_reward_0_5:.2f}")


--- Task 3.2: Running Epsilon-Greedy with 50% exploration (epsilon=0.5) ---

--- Running Epsilon-Greedy (epsilon=0.5) ---
Epsilon-Greedy simulation complete.
Total Cumulative Utility Score: 472.60
Cumulative Reward (epsilon=0.5): 472.60


### Summary of Task 3 Results

Comparing the Epsilon-Greedy performance with various exploration rates.

In [31]:
print("\n--- Epsilon-Greedy Performance Comparison for Task 3 ---")
print(f"Epsilon=0.01 Cumulative Utility: {eg_cumulative_reward_0_01:.2f}")
print(f"Epsilon=0.1 Cumulative Utility: {eg_cumulative_reward_0_1:.2f}")
print(f"Epsilon=0.5 Cumulative Utility: {eg_cumulative_reward_0_5:.2f}")

print("\nObservation: As epsilon (exploration rate) changes, the cumulative reward varies, highlighting the explore-exploit dilemma. For the tested exploration rates (10%, 1%, 50%), a moderate rate can offer a better balance between exploration and exploitation compared to very low or very high rates. Too little exploration (0.01) might miss better arms, while too much exploration (0.5) might waste too many pulls on suboptimal arms.")


--- Epsilon-Greedy Performance Comparison for Task 3 ---
Epsilon=0.01 Cumulative Utility: 510.80
Epsilon=0.1 Cumulative Utility: 493.80
Epsilon=0.5 Cumulative Utility: 472.60

Observation: As epsilon (exploration rate) changes, the cumulative reward varies, highlighting the explore-exploit dilemma. For the tested exploration rates (10%, 1%, 50%), a moderate rate can offer a better balance between exploration and exploitation compared to very low or very high rates. Too little exploration (0.01) might miss better arms, while too much exploration (0.5) might waste too many pulls on suboptimal arms.


## Summary of Results

We have successfully implemented and run three Multi-Armed Bandit algorithms: Epsilon-Greedy (with two epsilon values), UCB1, and Thompson Sampling. Below is a summary of their cumulative utility scores, which indicate the total reward gathered over 1000 patient treatments for each algorithm.

In [32]:
print("\n--- Algorithm Performance Summary ---")
print(f"Immediate Exploitation: Cumulative Utility = {ie_cumulative_reward:.2f}")
print(f"Epsilon-Greedy (epsilon=0.01): Cumulative Utility = {eg_cumulative_reward_0_01:.2f}")
print(f"Epsilon-Greedy (epsilon=0.1): Cumulative Utility = {eg_cumulative_reward_0_1:.2f}")
print(f"Epsilon-Greedy (epsilon=0.5): Cumulative Utility = {eg_cumulative_reward_0_5:.2f}")

algorithm_rewards = {
    "Immediate Exploitation": ie_cumulative_reward,
    "Epsilon-Greedy (epsilon=0.01)": eg_cumulative_reward_0_01,
    "Epsilon-Greedy (epsilon=0.1)": eg_cumulative_reward_0_1,
    "Epsilon-Greedy (epsilon=0.5)": eg_cumulative_reward_0_5
}
best_algorithm = max(algorithm_rewards, key=algorithm_rewards.get)
max_reward = algorithm_rewards[best_algorithm]
print(f"\nBased on cumulative utility, the best performing algorithm is {best_algorithm} with a total utility of {max_reward:.2f}.")


--- Algorithm Performance Summary ---
Immediate Exploitation: Cumulative Utility = 469.00
Epsilon-Greedy (epsilon=0.01): Cumulative Utility = 510.80
Epsilon-Greedy (epsilon=0.1): Cumulative Utility = 493.80
Epsilon-Greedy (epsilon=0.5): Cumulative Utility = 472.60

Based on cumulative utility, the best performing algorithm is Epsilon-Greedy (epsilon=0.01) with a total utility of 510.80.
